Самостоятельно реализуйте DecisionTree с некоторыми ограничениями (перечислены ниже). Задание может быть оценено частично.

Он должен получать на вход матрицу объект-признак и целевую переменную. На каждом этапе должен производить определение оптимальной пары признак-величина (для создания разделения данных, поступавших в вершину). Ограничение обучения реализуйте через ограничение допустимой максимальной глубины дерева - остальные опционарно.

**Рассмотрим игрушечную задачу бинарной классификации: поедет ли с Вами новый знакомый из бара? Это будет зависеть от Вашей внешности и красноречия, крепости предлагаемых напитков и, как это ни меркантильно, от количества потраченных в баре денег.**

### Создание набора данных

In [16]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

# Создание датафрейма с dummy variables
def create_df(dic, feature_list):
    out = pd.DataFrame(dic)
    return out

# Некоторые значения признаков есть в тесте, но нет в трейне и наоборот
def intersect_features(train, test):
    common_feat = list( set(train.keys()) & set(test.keys()))
    return train[common_feat], test[common_feat]

In [17]:
features = ['Внешность', 'Алкоголь_в_напитке',
            'Уровень_красноречия', 'Потраченные_деньги']
target = ['Поедет']

**Обучающая выборка**

In [18]:
df_train = {}
df_train['Внешность'] = ['приятная',
                         'приятная',
                         'приятная',
                         'отталкивающая',
                         'отталкивающая',
                         'отталкивающая',
                         'приятная'] 
df_train['Алкоголь_в_напитке'] = ['да', 
                                'да',
                                'нет',
                                'нет',
                                'да',
                                'да',
                                'да']
df_train['Уровень_красноречия'] = ['высокий',
                                   'низкий',
                                    'средний',
                                    'средний',
                                    'низкий',
                                   'высокий',
                                    'средний']
df_train['Потраченные_деньги'] = ['много',
                                 'мало',
                                  'много',
                                   'мало',
                                    'много',
                                  'много',
                                   'много']
df_train['Поедет'] = LabelEncoder().fit_transform([
                                                    '+',
                                                    '-',
                                                    '+',
                                                    '-',
                                                    '-',
                                                    '+',
                                                    '+'])

df_train = create_df(df_train, features)
df_train

,Внешность,Алкоголь_в_напитке,Уровень_красноречия,Потраченные_деньги,Поедет
0,приятная,да,высокий,много,0
1,приятная,да,низкий,мало,1
2,приятная,нет,средний,много,0
3,отталкивающая,нет,средний,мало,1
4,отталкивающая,да,низкий,много,1
5,отталкивающая,да,высокий,много,0
6,приятная,да,средний,много,0


**Тестовая выборка**

In [19]:
df_test = {}
df_test['Внешность'] = ['приятная', 'приятная', 'отталкивающая'] 
df_test['Алкоголь_в_напитке'] = ['нет', 'да', 'да']
df_test['Уровень_красноречия'] = ['средний', 'высокий', 'средний']
df_test['Потраченные_деньги'] = ['много', 'мало', 'много']
df_test = create_df(df_test, features)
df_test

,Внешность,Алкоголь_в_напитке,Уровень_красноречия,Потраченные_деньги
0,приятная,нет,средний,много
1,приятная,да,высокий,мало
2,отталкивающая,да,средний,много


In [20]:
# Некоторые значения признаков есть в тесте, но нет в трейне и наоборот
y = df_train['Поедет']
df_train, df_test = intersect_features(train=df_train, test=df_test)

df_train

,Алкоголь_в_напитке,Потраченные_деньги,Внешность,Уровень_красноречия
0,да,много,приятная,высокий
1,да,мало,приятная,низкий
2,нет,много,приятная,средний
3,нет,мало,отталкивающая,средний
4,да,много,отталкивающая,низкий
5,да,много,отталкивающая,высокий
6,да,много,приятная,средний


In [21]:
df_test

,Алкоголь_в_напитке,Потраченные_деньги,Внешность,Уровень_красноречия
0,нет,много,приятная,средний
1,да,мало,приятная,высокий
2,да,много,отталкивающая,средний


Для облегчения рекомендуется использовать (и реализовать) функции приведённые ниже.

In [27]:
# расчёт энтропии множества
def entropy(a_list):
    '''
    Calculates and returns entropy of given binary
    1d collection, using
    entropy = -p_0*log2(p_0) - p_1*log2(p_1),
    where
        p_0 - proportion of value 0 in a_list,
        p_1 - proportion of value 1 in a_list,
    '''
    a = np.array(a_list)
    if len(a) == 0:
         return 0 
    p_1 = a.sum() / len(a)
    p_0 = 1 - p_1 
    if a.sum() == 0 or len(a) == a.sum():
        return 0 
    entropy = -p_0 * np.log2(p_0) - p_1 * np.log2(p_1)
    return entropy

In [28]:
# расчет прироста информации

def information_gain(root, left, right):
    '''
    Calculates and returns Information Gain 
    if we split root by left and right
    '''
    return entropy(root) - 1/len(root)*(len(left)*entropy(left) + len(right) * entropy(right))

information_gain([0, 1, 0, 1, 1, 1, 1, 0], [0, 1, 0, 0], [1, 1, 1, 1])

np.float64(0.5487949406953987)

In [ ]:
# определение оптимального разбиения с точки зрения прироста информации

def best_feature_to_split(X, y, to_print=True):
    ''' Выводит прирост информации при разбиении по каждому признаку'''
    """\n Возвращает тройку (имя фичи, прирост информации, массив признаков) """
    best_splits = [] # (fname, ig, val)
    
    for feature in features:
        vals = X[feature].unique()
        pivot_keys = []
        for val in vals:
            mask = (X[feature] == val)
            pivot_key = (y[mask] == 1)
            pivot_keys.append((pivot_key.sum(), val))
        
        pivot_keys = sorted(pivot_keys)
        accum = set()
        best_ig = 0
        left_vals = []
        for pk in pivot_keys:
            accum.add(pk[1])
            mask = (X[feature].isin(accum))
            y_left = y[mask]
            y_right = y[~mask]
            ig = information_gain(y, y_left, y_right)
            if ig > best_ig:
                best_ig = ig
                left_vals = set(accum)
        best_splits.append((feature, best_ig, set(left_vals)))
    
    best_splits = sorted(best_splits, key = lambda x : x[1])
    if to_print:
        print(best_splits)
    
    return best_splits[-1]
best_feature_to_split(df_train, y)

[('Алкоголь_в_напитке', np.float64(0.0059777114237740125), {'нет'}), ('Внешность', np.float64(0.12808527889139443), {'приятная'}), ('Уровень_красноречия', np.float64(0.46956521111470706), {'средний', 'высокий'}), ('Потраченные_деньги', np.float64(0.46956521111470706), {'много'})]


('Потраченные_деньги', np.float64(0.46956521111470706), {'много'})

In [ ]:
class DecisionTree:
    """
    Decision tree, that considers binary features and solves
    binary classification task
    """
    class Node:
        """
        Node which are used to build DecisionTree
        """
        def __init__(self, is_leaf=False, val=None, feature=None, depth=None):
            """
            Initalizes a new tree node instance
            
            Parameters
            ----------
            is_leaf, Boolean
                defines whethes it's leaf node (has no children)
            val, int
                prediction value to be stored if it's leaf node
            feature, str
                name of feature which is used to split data
                by this node
            depth, int
                depth level of this node
            """
            self.left = None
            self.right = None
            self.is_leaf = is_leaf
            self.val = val
            self.feature = feature
            self.depth = depth
            self.major = None
            
                      
    def __init__(self, max_depth=3):
        """
        Initalizes a new DecisionTree instance
        """
        self.max_depth=max_depth
        self.root = None
        
    
    def build_tree(self, X, y, curr_node=None, curr_depth=0):
        """
        Returns recursively built classification tree
        """
        if curr_node is None:
            curr_node = self.Node()

        if curr_depth == self.max_depth or entropy(list(y)) == 0:
            curr_node.is_leaf = True 
            curr_node.major = int(round(y.mean()))
            return curr_node
        best_split = best_feature_to_split(X, y, to_print=False)
        if best_split[1] <= 0.0001:  
            curr_node.is_leaf = True
            curr_node.major = int(round(y.mean()))
            return curr_node
        mask = X[best_split[0]].isin(best_split[2])

        X_left = X[mask]
        X_right = X[~mask]
        y_left = y[mask]
        y_right = y[~mask]
        curr_node.val = best_split[2]
        curr_node.depth = curr_depth
        curr_node.feature = best_split[0]

        
        
        curr_node.is_leaf = False 
        curr_node.left = self.build_tree(X_left, y_left,curr_depth=curr_depth+1)
        curr_node.right = self.build_tree(X_right, y_right,curr_depth=curr_depth+1)
                        
        
        return curr_node
        
    def fit(self, X, y):
        """
        Fits to X: builds a decision tree that is
        appropriate to X
        Returns self.
        """
        self.root = self.build_tree(X, y)
        return self

    def predict(self, X):
        """
        Predicts binary target feature setting off from a tree
        that is built at 'fit' step
        Returns list of predicted values.
        """
        ...
        return y

In [73]:
tree_object = DecisionTree()

In [74]:
tree_object.fit(df_train, y)

In [75]:
tree_object.predict(df_test)

[0, 1, 1]